In [1]:
import sys, os
cmd = '''run.ipynb
--subset biology
--tree_version bottom-up
--llm_max_concurrent_calls 20
--num_eval_samples 1
'''
sys.argv = cmd.split()

## Imports

In [2]:
if os.path.basename(os.getcwd()) == 'src':
    %cd ..
    
    
%mkdir results trees
%pip install -r src/requirements.txt

c:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval


A subdirectory or file results already exists.
Error occurred while processing: results.
A subdirectory or file trees already exists.
Error occurred while processing: trees.


Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: 'torch>=2.6.0+cu124': Local version label can only be used with `==` or `!=` operators
    torch>=2.6.0+cu124
         ~~~~~~~^ (from line 14 of src/requirements.txt)


In [7]:
!git clone https://huggingface.co/datasets/quicktensor/lattice-bright-trees ./trees/BRIGHT

fatal: destination path './trees/BRIGHT' already exists and is not an empty directory.


In [3]:
#region Imports
import numpy as np
import pandas as pd
import pickle as pkl
import seaborn as sns
from datasets import load_dataset
from tqdm.auto import tqdm
import sys
import os
import logging
SRC_DIR = os.path.join(os.getcwd(), 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
from hyperparams import HyperParams
from tree_objects import SemanticNode, InferSample
from llm_apis import GenAIAPI, OpenAIResponsesAPI
from prompts import get_traversal_prompt_response_constraint, get_reranking_prompt
from utils import (
    setup_logger, 
    compute_node_registry,
    get_all_leaf_nodes_with_path, 
    get_node_id, 
    post_process, 
    save_exp, 
    load_exp,
    init_wandb_logging,
    finish_wandb_logging,
    wandb_log_iteration_metrics,
    wandb_log_reranking_metrics,
    wandb_log_final_summary,
    visualize_sample,
)
#endregion

#region Setup
hp = HyperParams.from_args()
BASE_DIR = os.getcwd()
RESULTS_DIR = f'{BASE_DIR}/results/BRIGHT/{hp.SUBSET}/'
os.makedirs(RESULTS_DIR, exist_ok=True)
logger = setup_logger('lattice_notebook', f"{RESULTS_DIR}/{hp}.log", logging.INFO)

# Initialize wandb logging
run_name = init_wandb_logging(hp, RESULTS_DIR, mode_override="disabled")
logger.info(f"Initialized wandb run: {run_name}")
#endregion

c:\Users\jmgil\Desktop\TESE\LATTICE\.venvLattice\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\jmgil\Desktop\TESE\LATTICE\.venvLattice\Lib\site-packages\pydantic\_internal\_generate_schema.py:2264: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\jmgil\Desktop\TESE\LATTICE\.venvLattice\Lib\site-packages\pydantic\_internal\_generate_schema.py:2264: UnsupportedFieldAttributeWarning:

Log file already exists: c:\Users\jmgil\Desktop\TESE\LATTICE\llm-guided-retrieval\results\BRIGHT\biology\S=biology-TV=bottom-up-TPV=5-RInTP=-1-NumLC=10-PlTau=5.0-RCF=0.5-LlmApiB=openai-Llm=gpt-4.1-NumI=20-NumES=1-MaxBS=2.log, appending to it.


## Data loading

In [9]:
#region Data loading
docs_df = pd.DataFrame(load_dataset('xlangai/BRIGHT', 'documents', split=hp.SUBSET))
examples_df = pd.DataFrame(load_dataset('xlangai/BRIGHT', 'examples', split=hp.SUBSET))
doc_id_to_content = {docs_df.iloc[i].id: docs_df.iloc[i].content for i in range(len(docs_df))}

tree_dict = pkl.load(open(f'{BASE_DIR}/trees/BRIGHT/{hp.SUBSET}/tree-{hp.TREE_VERSION}.pkl', 'rb'))
semantic_root_node = SemanticNode().load_dict(tree_dict) if isinstance(tree_dict, dict) else tree_dict
node_registry = compute_node_registry(semantic_root_node)
all_leaf_nodes = get_all_leaf_nodes_with_path(semantic_root_node)
doc_id_to_path = {get_node_id(leaf.id, docs_df): path for leaf, path in all_leaf_nodes}
#endregion

In [10]:
semantic_root_node.child[0]

ID: [0], Num children: 9, Description: This topic cluster provides a multi-disciplinary exploration of human and animal biology, behavior, and evolution. It covers fundamental life processes such as reproduction, development, and aging (senescence), as well as specific anatomical systems like the eye and joints. The collection delves into key evolutionary concepts like Cope's rule and speciation, and examines traits with both biological and cultural dimensions, including handedness, color perception, and kissing. It also provides a detailed overview of the genus *Homo*, focusing on humans and Neanderthals.

First 4 Children:

[0, [1], 9 children] This collection provides a comprehensive exploration of topics related to reproduction, development, and family dynamics in both humans and animals. It covers the biological and medical aspects of conception and birth, including twinning (monozygotic, dizygotic), superfecundation, parthenogenesis, and parturition. The node also delves into the

## Setup

In [11]:
assert os.environ.get('GOOGLE_API_KEY') is not None or os.environ.get('OPENAI_API_KEY') is not None , "Please set the GOOGLE_API_KEY or OPENAI environment variable."

In [12]:
#region Setup LLM API and Eval Samples
if hp.LLM_API_BACKEND == 'genai': 
    llm_api = GenAIAPI(hp.LLM, logger=logger, timeout=120, max_retries=4)
elif hp.LLM_API_BACKEND == 'openai':
    llm_api = OpenAIResponsesAPI(hp.LLM, logger=logger, timeout=hp.LLM_API_TIMEOUT, max_retries=hp.LLM_API_MAX_RETRIES)
else: raise ValueError(f'Unknown LM API backend: {hp.LLM_API_BACKEND}')

llm_api_kwargs = {
    'max_concurrent_calls': hp.LLM_MAX_CONCURRENT_CALLS,
    'response_mime_type': 'application/json',
    'response_schema': get_traversal_prompt_response_constraint(hp.TRAVERSAL_PROMPT_VERSION),
}

2026-04-25 12:15:28,699 - lattice_notebook - INFO - Initialized client for model: gpt-4.1
2026-04-25 12:15:29,114 - lattice_notebook - INFO - Initialized OpenAI Responses client with model: gpt-4.1


In [13]:
from utils import load_exp

if hp.LOAD_EXISTING:
  all_eval_samples, all_eval_metric_dfs = load_exp(RESULTS_DIR, hp, semantic_root_node, node_registry, logger)
  if len(all_eval_samples) > 0:
    logger.info(f'Loaded existing experiment with {len(all_eval_samples)} eval samples and {len(all_eval_metric_dfs)} eval metric dfs')
  if len(all_eval_samples) > 0:
    [sample.update_relevances(sample.prediction_tree) for sample in all_eval_samples]
    eval_metric_df = pd.DataFrame([sample.compute_eval_metrics(k=10) for sample in all_eval_samples])
    logger.info('; '.join([f'{k}: {eval_metric_df[k].mean():.2f}' for k in eval_metric_df.columns]))
else:
  all_eval_samples, all_eval_metric_dfs = [], []
  for i in range(min(examples_df.shape[0], hp.NUM_EVAL_SAMPLES)):
    sample = InferSample(
        semantic_root_node,
        node_registry,
        hp=hp,
        logger=logger,
        query=examples_df.iloc[i]['query'][:hp.MAX_QUERY_CHAR_LEN],
        gold_paths=[doc_id_to_path[docid] for docid in examples_df.iloc[i]['gold_ids']],
        excluded_ids_set=set(examples_df.iloc[i]['excluded_ids']),
        )
    all_eval_samples.append(sample)
  assert not any([sample.prediction_tree.excluded for sample in tqdm(all_eval_samples)])
logger.info('Hyperparams:\n'+'\n'.join([f'{k}:\t{v}' for k, v in vars(hp).items()]))
#endregion

100%|██████████| 1/1 [00:00<?, ?it/s]
2026-04-25 12:15:29,141 - lattice_notebook - INFO - Hyperparams:
dataset:	BRIGHT
subset:	biology
tree_version:	bottom-up
traversal_prompt_version:	5
reasoning_in_traversal_prompt:	-1
max_query_char_len:	None
max_doc_desc_char_len:	None
max_prompt_proto_size:	None
search_with_path_relevance:	True
num_leaf_calib:	10
pl_tau:	5.0
relevance_chain_factor:	0.5
llm_api_backend:	openai
llm:	gpt-4.1
llm_max_concurrent_calls:	20
llm_api_timeout:	120
llm_api_max_retries:	4
llm_api_staggering_delay:	0.1
num_iters:	20
num_eval_samples:	1
max_beam_size:	2
rerank:	False
load_existing:	False
num_threads:	8
suffix:	


## Retrieval loop

In [ ]:
for i in tqdm(range(len(all_eval_metric_dfs), hp.NUM_ITERS)):
    logger.info(f'-------------------- Iter {i} --------------------')
    
    inputs = [sample.get_step_prompts() for sample in all_eval_samples]
    indptr = np.cumsum([0, *[len(x) for x in inputs]])
    flat_inputs = [y for x in inputs for y in x]
    flat_prompts, flat_slates = list(zip(*flat_inputs))
    slates = [flat_slates[indptr[j]:indptr[j+1]] for j in range(len(inputs))]

    flat_responses = await llm_api.run_batch(flat_prompts, **llm_api_kwargs)
    flat_response_jsons = [post_process(output, return_json=True) for output in tqdm(flat_responses)]
    response_jsons = [flat_response_jsons[indptr[j]:indptr[j+1]] for j in range(len(inputs))]

    for sample, sample_slates, sample_response_jsons in tqdm(zip(all_eval_samples, slates, response_jsons), total=len(all_eval_samples), desc='Updating samples'):
      sample.update(sample_slates, sample_response_jsons)

    eval_metric_df = pd.DataFrame([sample.compute_eval_metrics(k=10) for sample in all_eval_samples])
    all_eval_metric_dfs.append(eval_metric_df)
    
    # Log metrics
    wandb_log_iteration_metrics(eval_metric_df, i)
    logger.info('; '.join([f'{k}: {eval_metric_df[k].mean():.2f}' for k in eval_metric_df.columns]))
    # save_exp(RESULTS_DIR, hp, llm_api, all_eval_samples, all_eval_metric_dfs, allow_overwrite=True)  
    logger.info('-'*50)

  0%|          | 0/20 [00:00<?, ?it/s]2026-04-25 12:15:29,163 - lattice_notebook - INFO - -------------------- Iter 0 --------------------
2026-04-25 12:15:29,170 - lattice_notebook - INFO - Running a batch of 1 prompts...
2026-04-25 12:15:29,171 - lattice_notebook - INFO - Concurrency limited to 20 parallel calls.
Processing batch: 100%|██████████| 1/1 [00:11<00:00, 11.46s/it, errors=0, active=0, completed=1, 429s=0, 503s=0]
2026-04-25 12:15:40,631 - lattice_notebook - INFO - ============================================================
2026-04-25 12:15:40,632 - lattice_notebook - INFO - BATCH PROCESSING SUMMARY REPORT
2026-04-25 12:15:40,633 - lattice_notebook - INFO - ============================================================
2026-04-25 12:15:40,635 - lattice_notebook - INFO - Total Duration: 11.46 seconds
2026-04-25 12:15:40,635 - lattice_notebook - INFO - Total Prompts: 1
2026-04-25 12:15:40,636 - lattice_notebook - INFO - Successful: 1 (100.0%)
2026-04-25 12:15:40,637 - lattice_

In [23]:
save_exp(RESULTS_DIR, hp, llm_api, all_eval_samples, all_eval_metric_dfs, allow_overwrite=True)

## Load results

In [32]:
all_eval_samples, all_eval_metric_dfs = load_exp(RESULTS_DIR, hp, semantic_root_node, node_registry, logger)

In [33]:
all_eval_metric_dfs[-1]

,nDCG@10,Recall@10,Recall@100,Recall@all,Coverage
0,65.524387,60.0,60.0,60.0,190


## Debug / visualize

In [34]:
from utils import visualize_sample

In [35]:
# i = np.random.randint(len(all_eval_samples))
i = 0 # change this to visualize a different sample
step = hp.NUM_ITERS # or any step <= NUM_ITERS, tree will be shown up to this step (iteration)
visualize_sample(all_eval_samples[i], save_path='../visualize_sample.html', max_step=step)

Saved plot HTML to ../visualize_sample.html


## Top Retrieved Docs

In [44]:
i = 0  # sample index
k = 10 # top-k retrieved documents

sample = all_eval_samples[i]
top_preds = sample.get_top_predictions(k)

top_docs = []
for rank, (node, score) in enumerate(top_preds, start=1):
    doc_id = get_node_id(node.id, docs_df)
    doc_row = docs_df.loc[docs_df['id'] == doc_id].iloc[0].to_dict()
    doc_row['retrieval_rank'] = rank
    doc_row['retrieval_score'] = score
    top_docs.append(doc_row)

top_docs_df = pd.DataFrame(top_docs)
top_docs_df

,id,content,retrieval_rank,retrieval_score
0,insects_attracted_to_light/Phototaxis_3.txt,Phototaxis in invertebrates[edit]\nJellyfish[e...,1,0.719221
1,insects_attracted_to_light/Phototaxis_0.txt,"\nPhototaxis is a kind of taxis, or locomotory...",2,0.448807
2,temperature_sensed/Transient_receptor_potentia...,Function[edit]\nTRP channels modulate ion entr...,3,0.325106
3,humans_more_adapted_to_light_mode_or_dark_mode...,"In animals[edit]\nA bearded dragon, a diurnal ...",4,0.321427
4,humans_more_adapted_to_light_mode_or_dark_mode...,\nDiurnality is a form of plant and animal beh...,5,0.296482
5,insects_attracted_to_light/Phototaxis_2_8.txt,. 2003). Individual RNAi depletion of both CSR...,6,0.283365
6,insects_attracted_to_light/Phototaxis_5_0.txt,See also[edit]\nWikimedia Commons has media re...,7,0.283192
7,insects_attracted_to_light/Phototaxis_2_9.txt,cilium is always younger than the posterior o...,8,0.282970
8,insects_attracted_to_light/Phototaxis_4.txt,Relation to magnetic fields[edit]\nUnder exper...,9,0.278444
9,insects_attracted_to_light/Phototaxis_2_10.txt,ransduction cascade alters the stroke pattern ...,10,0.276777


## Additional Reranking (optional)

In [45]:
from prompts import get_reranking_prompt

def get_sample_rerank_prompt(sample):
    return get_reranking_prompt(sample.query, [x.desc for x, _ in sample.get_top_predictions(100)], hp=hp, logger=logger, topk=10)

def process_sample_rerank_response(sample, response):
    ranking = post_process(response, return_json=True)['ranking']
    top_preds = [x for x, _ in sample.get_top_predictions(100)]
    for rank, idx in enumerate(ranking):
        if hasattr(top_preds[idx], 'inverse_rank'):
            if isinstance(top_preds[idx].inverse_rank, float):
                top_preds[idx].inverse_rank = [top_preds[idx].inverse_rank]
            top_preds[idx].inverse_rank.append(1/(rank+1))
        else:
            top_preds[idx].inverse_rank = [1/(rank+1)]

In [46]:
eval_samples = all_eval_samples
all_rerank_prompts, all_rerank_constraints = list(zip(*[get_sample_rerank_prompt(sample) for sample in eval_samples]))
all_rerank_responses = await llm_api.run_batch(all_rerank_prompts, max_concurrent_calls=10, response_mime_type='application/json', response_schema=all_rerank_constraints[0])

2026-04-25 18:08:51,332 - lattice_notebook - INFO - Running a batch of 1 prompts...
2026-04-25 18:08:51,336 - lattice_notebook - INFO - Concurrency limited to 10 parallel calls.
Processing batch: 100%|██████████| 1/1 [00:06<00:00,  6.79s/it, errors=0, active=0, completed=1, 429s=0, 503s=0]
2026-04-25 18:08:58,142 - lattice_notebook - INFO - ============================================================
2026-04-25 18:08:58,142 - lattice_notebook - INFO - BATCH PROCESSING SUMMARY REPORT
2026-04-25 18:08:58,143 - lattice_notebook - INFO - ============================================================
2026-04-25 18:08:58,145 - lattice_notebook - INFO - Total Duration: 6.81 seconds
2026-04-25 18:08:58,145 - lattice_notebook - INFO - Total Prompts: 1
2026-04-25 18:08:58,147 - lattice_notebook - INFO - Successful: 1 (100.0%)
2026-04-25 18:08:58,148 - lattice_notebook - INFO - Failed: 0
2026-04-25 18:08:58,149 - lattice_notebook - INFO - Total Error Occurrences: 0
2026-04-25 18:08:58,149 - lattice

In [47]:
for sample, response in zip(eval_samples, all_rerank_responses):
    try:
        process_sample_rerank_response(sample, response)
    except Exception as e:
        logger.error(f'Error processing rerank response for sample with query: {sample.query[:100]}.. Error: {e}')

In [48]:
rerank_eval_metric_df = pd.DataFrame([
    sample.compute_eval_metrics(
        k=10,
        rel_fn=(
            lambda x, base_rel_fn=sample.get_rel_fn(leaf=True): (
                np.mean(x.inverse_rank) if hasattr(x, 'inverse_rank') else 0,
                base_rel_fn(x),
            )
        ),
    )
    for sample in eval_samples
])
logger.info('After reranking: '+'; '.join([f'{k}: {rerank_eval_metric_df[k].mean():.2f}' for k in rerank_eval_metric_df.columns]))

2026-04-25 18:08:58,191 - lattice_notebook - INFO - After reranking: nDCG@10: 65.52; Recall@10: 60.00; Recall@100: 60.00; Recall@all: 60.00; Coverage: 190.00
